# Lecke 10 - MI ügynökök éles környezetben

Ebben a leckében megtanulod az **éles üzemre vonatkozó mintákat** MI-ügynökök számára a Microsoft Agent Framework (Python) használatával. Áttekintjük:

- **Megfigyelhetőség** — időmérés és naplózás hozzáadása az ügynökinterakciókhoz
- **Értékelés** — egy értékelő ügynök használata a válaszok minőségének pontozására
- **Költségkezelés** — stratégiák a tokenek optimalizálására és a modell kiválasztására

A forgatókönyv egy **utazási ügynök**, amely segít a felhasználóknak utazások tervezésében, melyre megfigyelés és értékelés van rétegződve.


## Beállítás


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Üzembe helyezési megfontolások

Moving AI agents from prototypes to production requires careful attention to three pillars:

1. **Megfigyelhetőség** — Szükséged van rálátásra arra, mit csinál az ügynök, mennyi ideig tart, és mely eszközöket hívja meg. Nyomon követés és naplózás nélkül a termelési hibák hibakeresése szinte lehetetlen.

2. **Értékelés** — Automatizált minőségellenőrzések biztosítják, hogy az ügynök válaszai idővel pontosak, teljesek és hasznosak maradjanak. Egy értékelő ügynök pontozhatja a válaszokat a meghatározott kritériumok alapján.

3. **Költségkezelés** — A tokenhasználat közvetlenül befolyásolja a költségeket. Olyan stratégiák, mint a promptoptimalizálás, modellválasztás és gyorsítótárazás segítenek kordában tartani a kiadásokat anélkül, hogy feláldoznák a minőséget.


## Megfigyelhető ügynök létrehozása

Definiáljuk az utazási eszközöket és időméréssel burkoljuk az ügynök hívását, hogy figyelemmel kísérhessük a késleltetést. Éles környezetben az OpenTelemetry-vel vagy egy hasonló tracing háttérrendszerrel integrálnád.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Értékelési minták

Egy gyakori gyakorlat az, hogy egy második ügynököt használnak **értékelőként**. Az értékelő pontozza az elsődleges ügynök válaszát előre meghatározott kritériumok alapján, mint például a teljesség, a pontosság és a segítőkészség.

Ez lehetővé teszi:
- Automatizált minőségkapuk, mielőtt a válaszok eljutnak a felhasználókhoz
- Regresszió észlelése, amikor a promptok vagy a modellek változnak
- Az ügynök teljesítményének folyamatos figyelése az idő során


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Költségkezelési stratégiák

A költségek ellenőrzése létfontosságú az éles AI-ügynökök számára. Íme a kulcsfontosságú stratégiák:

| Stratégia | Leírás |
|---|---|
| **Prompt optimalizálás** | Tartsa a rendszerutasításokat tömören. Távolítsa el a felesleges kontextust a bemeneti tokenek csökkentése érdekében. |
| **Modellválasztás** | Használjon kisebb, olcsóbb modelleket (pl. GPT-4o-mini) egyszerű feladatokhoz, mint például osztályozás vagy kinyerés, és tartsa fenn a nagyobb modelleket a komplexebb következtetésekhez. |
| **Gyorsítótárazás** | Gyorsítótárazza az eszközök eredményeit és a gyakori lekérdezéseket, hogy elkerülje a felesleges API-hívásokat. |
| **Token költségvetések** | Állítson be `max_tokens` korlátokat, hogy megelőzze a váratlanul hosszú válaszokat. |
| **Csoportosítás** | Szükség esetén csoportosítsa több felhasználói lekérdezést egyetlen API-hívásba. |

Gyakorlatban jól működik egy réteges megközelítés: irányítsa az egyszerű kéréseket egy gyors, olcsó modellhez, és csak a bonyolultabb lekérdezéseket továbbítsa egy erősebb modellhez.


## Összefoglalás

Ebben a leckében megtanultad, hogyan:

1. **Megfigyelhetőséget hozzáadni** az ügynök-interakciókhoz időzítéssel és naplózással, megteremtve a nyomkövetés és a monitorozás alapjait.
2. **Az ügynök válaszait automatikusan értékelni** egy értékelő ügynök használatával, amely pontozza a teljességet, a pontosságot és a hasznosságot.
3. **A költségeket kezelni** prompt optimalizálással, modellválasztással, gyorsítótárazással és token költségkeretekkel.

Ezek az üzemeltetési minták segítenek biztosítani, hogy az AI-ügynökeid nagy léptékben megbízhatóak, mérhetők és költséghatékonyak legyenek.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Felelősségkizárás:
Ezt a dokumentumot az AI-alapú fordító szolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordították. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. A dokumentum eredeti nyelvű változatát kell tekinteni irányadónak. Kritikus információk esetén szakember által végzett emberi fordítást javaslunk. Nem vállalunk felelősséget az ebből a fordításból eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
